In [2]:
import os
import sys
import django


PROJEKT_DIR = os.path.abspath(".")
if PROJEKT_DIR not in sys.path:
    sys.path.insert(0, PROJEKT_DIR)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "locallibrary.settings")


os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()  

from catalog.models import Author, Genre, Language, Book, BookInstance
print("django.setup() OK — Django", django.get_version())

django.setup() OK — Django 5.2.15


In [7]:
# get_or_create gibt (object, created_bool) zurück
# save() ist auch mit drin 
from datetime import date

sf, _ = Genre.objects.get_or_create(name = "Science-Fiction")
fant, _ = Genre.objects.get_or_create(name="Fantasy")
roman, _ = Genre.objects.get_or_create(name="Roman")

# Sprachen
de, _ = Language.objects.get_or_create(name="Deutsch")
en, _ = Language.objects.get_or_create(name="Englisch")

# Autor:innen
tolkien, _ = Author.objects.get_or_create(
    first_name="J. R. R.", last_name="Tolkien",
    defaults={"date_of_birth": date(1892, 1, 3)},
)
lem, _ = Author.objects.get_or_create(
    first_name="Stanisław", last_name="Lem",
    defaults={"date_of_birth": date(1921, 9, 12)},
)
le_guin, _ = Author.objects.get_or_create(
    first_name="Ursula K.", last_name="Le Guin",
    defaults={"date_of_birth": date(1929, 10, 21)},
)

# Bücher (ISBN ist unique -> idempotenter Match-Schlüssel)
hobbit, _ = Book.objects.get_or_create(
    isbn="9780261103283",
    defaults={"title": "Der Hobbit", "author": tolkien,
              "language": de, "summary": "Bilbo Beutlin auf Abenteuerreise."},
)
solaris, _ = Book.objects.get_or_create(
    isbn="9780156027601",
    defaults={"title": "Solaris", "author": lem,
              "language": de, "summary": "Erstkontakt mit einem Ozean-Planeten."},
)
darkness, _ = Book.objects.get_or_create(
    isbn="9780441478125",
    defaults={"title": "The Left Hand of Darkness", "author": le_guin,
              "language": en, "summary": "Geschlecht und Gesellschaft auf Gethen."},
)

# M2M -> set() ... Keine Duplikate, Reihenfolge egal, idempotent
hobbit.genre.set([fant])
hobbit.genre.add(fant)
solaris.genre.set([sf, roman])
solaris.genre.all()

BookInstance.objects.get_or_create(
    book=hobbit,
    defaults={"status": "a"},
)
BookInstance.objects.get_or_create(
    book=solaris, 
    defaults={"status": "o", "due_back": date(2026, 6, 15)},
)
BookInstance.objects.get_or_create(
    book=darkness, 
    defaults={"status": "a"},
)

print("Bücher:", Book.objects.count(),
      "| Autor:innen:", Author.objects.count(),
      "| Exemplare:", BookInstance.objects.count())



Bücher: 3 | Autor:innen: 3 | Exemplare: 3


In [ ]:
all_books = Book.objects.all() # noch keine DB 
all_books

# erst hier 
print(all_books.count()) # SELECT COUNT(*) 

for b in all_books:
    print("-", b.title, "/", b.author )
# Reihenfolge aus Meta.ordering = ["title"]

3
- Der Hobbit / Tolkien, J. R. R.
- Solaris / Lem, Stanisław
- The Left Hand of Darkness / Le Guin, Ursula K.


In [13]:
print(list(Book.objects.all().filter(title__icontains="solaris")))
print(list(Book.objects.exclude(title__icontains="solaris"))) # Komplement zu filter()

# Numerischer Lookup
print(list(Author.objects.filter(date_of_birth__year__gte=1921)))

[<Book: Solaris>]
[<Book: Der Hobbit>, <Book: The Left Hand of Darkness>]
[<Author: Le Guin, Ursula K.>, <Author: Lem, Stanisław>]


In [17]:
str(Author.objects.filter(date_of_birth__year__gte=1921).query)


'SELECT "catalog_author"."id", "catalog_author"."first_name", "catalog_author"."last_name", "catalog_author"."date_of_birth", "catalog_author"."date_of_death" FROM "catalog_author" WHERE "catalog_author"."date_of_birth" >= 1921-01-01 ORDER BY "catalog_author"."last_name" ASC, "catalog_author"."first_name" ASC'

In [24]:
from django.db.models import Q

qs = Book.objects.filter(Q(title__icontains="hobbit") | Q(language__name="Englisch"))
print(list(qs))

# (SF ODER Fantasy) UND nicht englischsprachig
qs2 = Book.objects.filter((Q(genre__name="Science-Fiction") | Q(genre__name="Fantasy")) & ~Q(language__name="Englisch"))
print(list(qs2))


[<Book: Der Hobbit>, <Book: The Left Hand of Darkness>]
[<Book: Der Hobbit>, <Book: Solaris>]


In [31]:
#Book.objects.get(isbn="0000000000000")

from django.shortcuts import get_object_or_404
from django.http import Http404
try:
    get_object_or_404(Book, isbn="0000000000000")
except Http404:
    print("-> 404")
#get_object_or_404(Book, isbn="0000000000000")

-> 404
